## Downloading Vild model from GCloud

In [11]:
!gcloud auth login

Go to the following link in your browser, and complete the sign-in prompts:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=32555940559.apps.googleusercontent.com&redirect_uri=https%3A%2F%2Fsdk.cloud.google.com%2Fauthcode.html&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fappengine.admin+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcompute+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Faccounts.reauth&state=hAnW9TYkrpXiGndh7LkO7LzCnRaQ32&prompt=consent&token_usage=remote&access_type=offline&code_challenge=dAuX0Oq4fsG2H8yDehXNIRcGdzdyXWWQgbJKcErlhrU&code_challenge_method=S256

Once finished, enter the verification code provided in your browser: 4/0Aci98E_ApmgyfiBDDuW1SksFtHr6UwWKlXnHb8Xb7WNO2DlQXEdnplz1qkS1PB3aU1SsTQ

You are now logged in as [shinde.shantanu21@gmail.com].
Your current pr

In [12]:
!gcloud storage --no-user-output-enabled --verbosity error cp --recursive gs://cloud-tpu-checkpoints/detection/projects/vild/colab/image_path_v2 ./

## Unzipping and loading the COCO Subset

In [13]:
!unzip -q /content/coco_subset_100.zip

replace coco_subset_100/val2017/000000501005.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: a
error:  invalid response [a]
replace coco_subset_100/val2017/000000501005.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: A


In [14]:
import json

In [15]:
with open('/content/coco_subset_100/annotations/instances_train2017.json', 'r') as js:
    annot = json.load(js)

In [16]:
class_map = {cat['id']:cat['name'] for cat in annot['categories']}
classes = list(class_map.values())

In [17]:
from collections import defaultdict

In [18]:
images = {img['id']:img['file_name'] for img in annot['images']}
annotations = defaultdict(list)
for an in annot['annotations']:
    annotations[an['image_id']].append([an['bbox'], class_map[an['category_id']]])

## Installing and loading CLIP model

In [19]:
!pip -q install git+https://github.com/openai/CLIP.git

  Preparing metadata (setup.py) ... done


In [20]:
import clip

clip.available_models()
model, preprocess = clip.load("ViT-B/32")

100%|███████████████████████████████████████| 338M/338M [00:03<00:00, 98.3MiB/s]


## Create text embeddings from the categories in the dataset

In [21]:
import torch
from tqdm import tqdm

In [22]:
import numpy as np

In [23]:
single_template = [
    'a photo of {article} {}.'
]

def article(name):
  return 'an' if name[0] in 'aeiou' else 'a'

def processed_name(name, rm_dot=False):
  res = name.replace('_', ' ').replace('/', ' or ').lower()
  if rm_dot:
    res = res.rstrip('.')
  return res

def build_text_embedding(categories):
  templates = single_template

  run_on_gpu = torch.cuda.is_available()

  with torch.no_grad():
    all_text_embeddings = []
    print('Building text embeddings...')
    for category in tqdm(categories):
      texts = [
        template.format(processed_name(category['name'], rm_dot=True),
                        article=article(category['name']))
        for template in templates]
      texts = clip.tokenize(texts) #tokenize
      if run_on_gpu:
        texts = texts.cuda()
      text_embeddings = model.encode_text(texts) #embed with text encoder
      text_embeddings /= text_embeddings.norm(dim=-1, keepdim=True)
      text_embedding = text_embeddings.mean(dim=0)
      text_embedding /= text_embedding.norm()
      all_text_embeddings.append(text_embedding)
    all_text_embeddings = torch.stack(all_text_embeddings, dim=1)
    if run_on_gpu:
      all_text_embeddings = all_text_embeddings.cuda()
  return all_text_embeddings.cpu().numpy().T

In [24]:
category_names = ["background"] + classes
categories = [{'name': item, 'id': idx+1,} for idx, item in enumerate(category_names)]
text_features = build_text_embedding(categories)


Building text embeddings...


100%|██████████| 81/81 [00:01<00:00, 68.21it/s] 


## Get predictions from Vild model and match it with ground truth in Coco dataset

In [25]:
def nms(dets, scores, thresh, max_dets=1000):
  """Non-maximum suppression.
  Args:
    dets: [N, 4]
    scores: [N,]
    thresh: iou threshold. Float
    max_dets: int.
  """
  y1 = dets[:, 0]
  x1 = dets[:, 1]
  y2 = dets[:, 2]
  x2 = dets[:, 3]

  areas = (x2 - x1) * (y2 - y1)
  order = scores.argsort()[::-1]

  keep = []
  while order.size > 0 and len(keep) < max_dets:
    i = order[0]
    keep.append(i)

    xx1 = np.maximum(x1[i], x1[order[1:]])
    yy1 = np.maximum(y1[i], y1[order[1:]])
    xx2 = np.minimum(x2[i], x2[order[1:]])
    yy2 = np.minimum(y2[i], y2[order[1:]])

    w = np.maximum(0.0, xx2 - xx1)
    h = np.maximum(0.0, yy2 - yy1)
    intersection = w * h
    overlap = intersection / (areas[i] + areas[order[1:]] - intersection + 1e-12)

    inds = np.where(overlap <= thresh)[0]
    order = order[inds + 1]
  return keep

In [26]:
import tensorflow.compat.v1 as tf

In [27]:
image_dir = '/content/coco_subset_100/train2017'

In [28]:
from os import path

In [29]:
session = tf.Session(graph=tf.Graph())
saved_model_dir = './image_path_v2'

_ = tf.saved_model.loader.load(session, ['serve'], saved_model_dir)

Instructions for updating:
Use `tf.saved_model.load` instead.


In [30]:
max_boxes_to_draw = 300
iou_threshold = 0.3

In [31]:
def compute_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    inter = max(0, x2 - x1) * max(0, y2 - y1)

    area1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    area2 = (box2[2]-box2[0]) * (box2[3]-box2[1])

    union = area1 + area2 - inter

    return inter / union if union > 0 else 0

In [32]:
final_dict = defaultdict(list)
for img_id in images:
    image_path = path.join(image_dir, images[img_id])

    roi_boxes, roi_scores, detection_boxes, scores_unused, box_outputs, detection_masks, visual_features, image_info = session.run(
        ['RoiBoxes:0', 'RoiScores:0', '2ndStageBoxes:0', '2ndStageScoresUnused:0', 'BoxOutputs:0', 'MaskOutputs:0', 'VisualFeatOutputs:0', 'ImageInfo:0'],
        feed_dict={'Placeholder:0': [image_path,]})
    roi_boxes = np.squeeze(roi_boxes, axis=0)  # squeeze
    # no need to clip the boxes, already done
    roi_scores = np.squeeze(roi_scores, axis=0)

    detection_boxes = np.squeeze(detection_boxes, axis=(0, 2))
    scores_unused = np.squeeze(scores_unused, axis=0)
    box_outputs = np.squeeze(box_outputs, axis=0)
    detection_masks = np.squeeze(detection_masks, axis=0)
    visual_features = np.squeeze(visual_features, axis=0)

    image_info = np.squeeze(image_info, axis=0)  # obtain image info
    image_scale = np.tile(image_info[2:3, :], (1, 2))
    image_height = int(image_info[0, 0])
    image_width = int(image_info[0, 1])

    #rescaled_detection_boxes = detection_boxes #/ image_scale # rescale
    scale_y, scale_x = image_info[2]
    offset_y, offset_x = image_info[3] if image_info.shape[0] > 3 else (0, 0)

    rescaled_detection_boxes = detection_boxes.copy()

    # remove scale
    rescaled_detection_boxes[:, 0] = (rescaled_detection_boxes[:, 0] - offset_y) / scale_y
    rescaled_detection_boxes[:, 2] = (rescaled_detection_boxes[:, 2] - offset_y) / scale_y
    rescaled_detection_boxes[:, 1] = (rescaled_detection_boxes[:, 1] - offset_x) / scale_x
    rescaled_detection_boxes[:, 3] = (rescaled_detection_boxes[:, 3] - offset_x) / scale_x

    nmsed_indices = nms(
      detection_boxes,
      roi_scores,
      thresh=0.6
      )

    box_sizes = (rescaled_detection_boxes[:, 2] - rescaled_detection_boxes[:, 0]) * (rescaled_detection_boxes[:, 3] - rescaled_detection_boxes[:, 1])

    valid_indices = np.where(
        np.logical_and(
            np.isin(np.arange(len(roi_scores), dtype=int), nmsed_indices),
            np.logical_and(
                np.logical_not(np.all(roi_boxes == 0., axis=-1)),
                np.logical_and(
                roi_scores >= 0.9,
                box_sizes > 220
                )
            )
        )
    )[0]

    # Extract valid boxes, and pick max required boxes from that
    detection_roi_scores = roi_scores[valid_indices][:max_boxes_to_draw, ...]
    detection_boxes = detection_boxes[valid_indices][:max_boxes_to_draw, ...]
    detection_masks = detection_masks[valid_indices][:max_boxes_to_draw, ...]
    detection_visual_feat = visual_features[valid_indices][:max_boxes_to_draw, ...]
    rescaled_detection_boxes = rescaled_detection_boxes[valid_indices][:max_boxes_to_draw, ...]

    roi_boxes_all = roi_boxes[valid_indices][:max_boxes_to_draw, ...]
    roi_scores_all = detection_roi_scores
    scores_all = detection_visual_feat.dot(text_features.T)

    class_ids = np.argmax(scores_all, axis=1)
    scores = np.max(scores_all, axis=1)

    ymin, xmin, ymax, xmax = np.split(rescaled_detection_boxes, 4, axis=-1)
    pred_boxes = np.concatenate([xmin, ymin, xmax, ymax], axis=-1)

    # Match the predictions with ground truth
    rows = []
    used_gt = set()
    for i in range(len(pred_boxes)):
        if class_ids[i] == 0:
            continue

        if scores[i] < 0.2:
            continue

        pred_class = category_names[class_ids[i]]
        pred_box = pred_boxes[i]
        score = scores[i]
        roi_score = roi_scores[i]
        roi_box = roi_boxes[i]

        gt_boxes = []
        gt_classes = []

        for entry in annotations[img_id]:
            x, y, w, h = entry[0]
            gt_boxes.append([x, y, x+w, y+h])
            gt_classes.append(entry[1])
        best_iou = 0
        best_j = -1
        for j, gt_box in enumerate(gt_boxes):
            if j in used_gt:
                continue
            iou = compute_iou(pred_box, gt_box)
            if iou > best_iou:
                best_iou = iou
                best_j = j

        if best_iou >= iou_threshold:
            gt_box_match = gt_boxes[best_j]
            gt_class_match = gt_classes[best_j]
            used_gt.add(best_j)
        else:
            gt_box_match = None
            gt_class_match = None
        rows.append({
            'pred_box': pred_box,
            'pred_class' : pred_class,
            'pred_score' : score,
            'pred_roi_box' : roi_box,
            'pred_roi_score' : roi_score,
            'ground_box' : gt_box_match,
            'ground_class' : gt_class_match,
        })
    final_dict[img_id] = rows

## Save the results to JSON file

In [33]:
def convert(o):
    if isinstance(o, np.ndarray):
        return o.tolist()
    elif isinstance(o, (np.int64, np.int32)):
        return int(o)
    elif isinstance(o, (np.float32, np.float64)):
        return float(o)
    else:
        raise TypeError(f"Object of type {type(o)} is not JSON serializable")

with open('results.json', 'w') as results:
    json.dump(final_dict, results, default=convert, indent=4)